# VerseLens: Multi-Task Poetry Analysis with DeBERTa-v3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pannagaJois/verselens/blob/main/verselens.ipynb)

VerseLens is a multi-task learning pipeline for poetry analysis, classification (theme, affect, form), and interpretability using Integrated Gradients.

> **Runtime recommendation:** Use a GPU runtime (**Runtime > Change runtime type > T4 GPU**).

### Cell 1: Install Dependencies

In [ ]:
# ===== CELL 1 =====
!pip install -q -U transformers sentencepiece accelerate scikit-learn


### Cell 2: Imports, Seed & Device Setup

In [ ]:
# ===== CELL 2 =====
import os, re, ast, json, math, random, shutil, copy, glob
from collections import Counter
from typing import Dict, List, Optional, Tuple, Union

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

# Select AMP dtype: use bfloat16 if GPU architecture supports it (Ampere+ e.g. A100/L4); fallback to float16 (Turing e.g. T4)
BF16_AVAILABLE = USE_AMP and hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16_AVAILABLE else torch.float16
USE_SCALER = USE_AMP and (AMP_DTYPE == torch.float16)


def autocast_ctx():
    """Modern context manager for mixed precision across PyTorch versions."""
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP)
    return torch.cuda.amp.autocast(enabled=USE_AMP)


def set_seed(seed: int = 42) -> None:
    """Seed python, numpy and torch (CPU + CUDA) for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)
print(f"Device: {DEVICE.type}")
if USE_AMP:
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name} | AMP dtype: {AMP_DTYPE} | GradScaler enabled: {USE_SCALER}")
else:
    print("WARNING: no GPU found. Runtime > Change runtime type > T4 GPU.")


### Cell 3: Clone Repository (if needed) & Discover CSV Datasets

In [ ]:
# ===== CELL 3 =====
REPO_URL = "https://github.com/pannagaJois/verselens.git"
# If datasets folder is neither in pwd nor under verselens, clone the repo
![ -d datasets ] || [ -d verselens ] || git clone -q {REPO_URL}

CSV_PATHS = sorted(
    glob.glob("datasets/**/*.csv", recursive=True) +
    glob.glob("datasets/*.csv") +
    glob.glob("verselens/datasets/**/*.csv", recursive=True) +
    glob.glob("verselens/datasets/*.csv")
)
if not CSV_PATHS:  # fallback: search current directory and all subfolders
    CSV_PATHS = sorted(glob.glob("**/*.csv", recursive=True))
CSV_PATHS = sorted(list(set(CSV_PATHS)))
assert CSV_PATHS, "No CSV files found after checking pwd and cloning. Check datasets folder."

print(f"Found {len(CSV_PATHS)} CSV files:")
for p in CSV_PATHS:
    print(f"  {os.path.getsize(p) / 1e6:8.2f} MB  {p}")


def find_csv(name: str) -> Optional[str]:
    """Return the first CSV path whose basename equals `name`, else None."""
    for p in CSV_PATHS:
        if os.path.basename(p) == name:
            return p
    return None


### Cell 4: Inspect Datasets & Target Columns

In [ ]:
# ===== CELL 4 =====
INSPECT_NAME = "poetry_train_cleaned.csv"  # <-- CHANGE to inspect another file (e.g. POCAAffectClassification_cleaned.csv)

# Header-only peek at every CSV so you can see which columns exist where
print("=== Columns of every CSV ===")
for p in CSV_PATHS:
    try:
        print(f"{os.path.basename(p)}: {list(pd.read_csv(p, nrows=0).columns)}")
    except Exception as e:  # noqa: BLE001
        print(f"{os.path.basename(p)}: could not read header ({e})")

inspect_path = find_csv(INSPECT_NAME)
assert inspect_path, f"'{INSPECT_NAME}' not found. Pick one of the names printed in Cell 3."
_df = pd.read_csv(inspect_path)

print(f"\n=== {INSPECT_NAME} ===")
print("Shape:", _df.shape)
print("Columns:", list(_df.columns))

print("\n--- One sample row (transposed, values truncated) ---")
print(_df.iloc[0].astype(str).str.slice(0, 200).to_frame("value"))

print("\n--- Null counts ---")
print(_df.isnull().sum().to_frame("nulls"))

print("\n--- Per-column summary (use this to decide list-like vs binary labels) ---")
for c in _df.columns:
    ex = str(_df[c].dropna().iloc[0])[:60] if _df[c].notna().any() else "NA"
    print(f"{c:25s} dtype={str(_df[c].dtype):10s} n_unique={_df[c].nunique():7d} example={ex!r}")


### Cell 5: Model & Training Configuration

In [ ]:
# ===== CELL 5 =====
CONFIG = {
    "backbone": "microsoft/deberta-v3-base",
    "csv_file": "poetry_train_cleaned.csv",   # <-- CHANGE: file basename from Cell 3 listing
    "max_len": 512,
    "batch_size": 8,
    "grad_accum": 2,                          # effective batch = 16; fits a T4
    "lr_backbone": 2e-5,
    "lr_heads": 1e-4,
    "weight_decay": 0.01,
    "epochs": 4,                              # <-- CHANGE as needed
    "warmup_ratio": 0.1,
    "seed": 42,
    "val_split": 0.1,
    "threshold": 0.5,                         # decision threshold for F1 and analyze()
    "debug_rows": None,                       # <-- CHANGE: e.g. 2000 for a quick smoke test, None = all rows
    "min_label_count": 5,                     # list-like mode: drop labels rarer than this in train
    "max_labels": 50,                         # list-like mode: keep at most this many labels per task
    "text_col": "poem",                       # <-- VERIFY against Cell 4 output
    # task name -> column name (str: list-like OR single 0/1 column)
    #              or list of column names (multiple binary 0/1 columns)
    # Tasks whose columns are missing are skipped with a [skip] message.
    "label_cols": {
        "theme": "labels",                    # <-- Matches 'labels' in poetry_train_cleaned.csv
        # Examples to add/swap (POCA file has: labels, label_names, scores):
        # "emotion": "label_names",           # <-- needs csv_file = POCAAffectClassification_cleaned.csv
        # "form": ["sonnet", "haiku"],        # <-- binary 0/1 columns mode
    },
    "task_weights": {                         # <-- CHANGE: lambda_t per task (default 1.0 if absent)
        "theme": 1.0,
    },
    "pos_weight_cap": 5.0,
}
# Future work (not implemented here):
#  - Line-level emotion from poemo.english.emotion_cleaned.csv (needs a token-classification head)
#  - Summarization from poemsum_*.csv
#  - Sliding-window chunking for poems longer than 512 tokens (see Cell 7)
#  - POCA `scores` column as soft labels (replace BCE targets with normalized scores)
set_seed(CONFIG["seed"])
print(json.dumps(CONFIG, indent=2))


### Cell 6: Label Builder & Dataset Preparation

In [ ]:
# ===== CELL 6 =====
def parse_listlike(x) -> List[str]:
    """Parse '[a, b]', "['a' 'b']", "['a','b']", 'a;b', 'a|b', real lists, or a single string into lowercase labels."""
    if isinstance(x, (list, tuple, set, np.ndarray)):
        items = list(x)
    else:
        if x is None or pd.isna(x):
            return []
        s = str(x).strip()
        if s == "" or s.lower() in {"nan", "none", "null", "[]"}:
            return []
        items = None
        # Handles numpy string representation e.g. "['Religion' 'Living']" and standard "['Religion', 'Living']"
        quoted = re.findall(r"['\"]([^'\"]+)['\"]", s)
        if quoted:
            items = quoted
        elif s[0] in "[(":
            try:
                v = ast.literal_eval(s)
                items = list(v) if isinstance(v, (list, tuple, set)) else [v]
            except (ValueError, SyntaxError):
                items = None  # unquoted like [sadness, beauty] -> fall through
        if items is None:
            items = re.split(r"[,;|]", s.strip("[]()"))
    out = []
    for it in items:
        t = str(it).strip().strip('\'"').strip().lower()
        if t and t not in {"nan", "none"}:
            out.append(t)
    return out


def build_labels(
    df: pd.DataFrame,
    spec: Union[str, List[str]],
    vocab: Optional[List[str]] = None,
    min_count: int = 1,
    max_labels: int = 10_000,
) -> Optional[Tuple[np.ndarray, np.ndarray, List[str], str]]:
    """Return (Y[N,L], mask[N], vocab, mode) or None if the task cannot be built.
    Auto-detects: list of columns -> binary; single 0/1 column -> binary; otherwise list-like parsing.
    If `vocab` is given (validation split), it is reused instead of rediscovered."""
    # ---- mode (b): several binary columns ----
    if isinstance(spec, (list, tuple)):
        cols = list(vocab) if vocab is not None else [c for c in spec if c in df.columns]
        for c in spec:
            if c not in df.columns:
                print(f"    [skip] column '{c}' not in CSV")
        if not cols:
            return None
        num = df[cols].apply(pd.to_numeric, errors="coerce")
        mask = num.notna().any(axis=1).values.astype(np.float32)
        Y = (num.fillna(0).values > 0.5).astype(np.float32)
        return Y, mask, cols, "binary-columns"

    # ---- single column ----
    col = spec
    if col not in df.columns:
        if col == "label" and "labels" in df.columns:
            col = "labels"
        elif col == "labels" and "label" in df.columns:
            col = "label"
        else:
            print(f"    [skip] column '{col}' not in CSV")
            return None
    series = df[col]
    nonnull = series.dropna()
    vals = pd.to_numeric(nonnull, errors="coerce")
    if len(vals) > 0 and vals.notna().all() and set(vals.unique()) <= {0, 1, 0.0, 1.0}:
        Y = pd.to_numeric(series, errors="coerce").fillna(0).values.reshape(-1, 1).astype(np.float32)
        mask = series.notna().values.astype(np.float32)
        return Y, mask, [col], "binary-single-column"

    # ---- mode (a): list-like strings ----
    parsed = series.map(parse_listlike)
    mask = series.notna().values.astype(np.float32)
    if vocab is None:
        cnt = Counter(l for row, m in zip(parsed, mask) if m > 0 for l in row)
        vocab = [l for l, c in cnt.most_common(max_labels) if c >= min_count]
    if not vocab:
        return None
    idx = {l: i for i, l in enumerate(vocab)}
    Y = np.zeros((len(df), len(vocab)), dtype=np.float32)
    for i, row in enumerate(parsed):
        for l in row:
            if l in idx:
                Y[i, idx[l]] = 1.0
    return Y, mask, vocab, "list-like"


csv_path = find_csv(CONFIG["csv_file"])
assert csv_path, f"CONFIG['csv_file']={CONFIG['csv_file']!r} not found. Check Cell 3 listing!"
df = pd.read_csv(csv_path)
text_col = CONFIG["text_col"]
assert text_col in df.columns, f"text_col {text_col!r} not in columns {list(df.columns)}. Fix CONFIG['text_col'] using Cell 4 output!"

df = df.dropna(subset=[text_col]).copy()
df[text_col] = df[text_col].astype(str).str.strip()
df = df[df[text_col].str.len() > 0]
if CONFIG["debug_rows"]:
    df = df.sample(n=min(CONFIG["debug_rows"], len(df)), random_state=CONFIG["seed"])
df = df.reset_index(drop=True)

train_df, val_df = train_test_split(df, test_size=CONFIG["val_split"], random_state=CONFIG["seed"], shuffle=True)
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)
print(f"Train: {len(train_df)} Val: {len(val_df)}")

TASK_COLS: Dict[str, Union[str, List[str]]] = {}
TASK_VOCAB: Dict[str, List[str]] = {}
TRAIN_Y, TRAIN_M, VAL_Y, VAL_M = {}, {}, {}, {}
POS_WEIGHTS: Dict[str, torch.Tensor] = {}

for task, spec in CONFIG["label_cols"].items():
    print(f"\nTask '{task}' <- {spec}")
    res = build_labels(train_df, spec, None, CONFIG["min_label_count"], CONFIG["max_labels"])
    if res is None:
        print(f"[skip] task '{task}': could not build labels")
        continue
    Ytr, Mtr, vocab, mode = res
    if Mtr.sum() == 0:
        print(f"[skip] task '{task}': no labeled training rows")
        continue
    Yv, Mv, _, _ = build_labels(val_df, spec, vocab=vocab)
    TASK_COLS[task], TASK_VOCAB[task] = spec, vocab
    TRAIN_Y[task], TRAIN_M[task], VAL_Y[task], VAL_M[task] = Ytr, Mtr, Yv, Mv

    pos = (Ytr * Mtr[:, None]).sum(0)
    neg = Mtr.sum() - pos
    pw = np.minimum(neg / np.maximum(pos, 1.0), CONFIG["pos_weight_cap"])
    POS_WEIGHTS[task] = torch.tensor(pw, dtype=torch.float32)

    shown = vocab[:12]
    print(f"{task}: {len(vocab)} labels -> {shown}{' ...' if len(vocab) > 12 else ''}  (mode={mode})")
    print(f"    labeled rows train={int(Mtr.sum())} val={int(Mv.sum())}")
    print(f"    pos_weight (cap {CONFIG['pos_weight_cap']}): {np.round(pw[:12], 2).tolist()}")

assert TASK_COLS, "No tasks built — check CONFIG['label_cols'] against Cell 4 output!"
print(f"\nActive tasks: {list(TASK_COLS)}")


### Cell 7: Dataset & DataLoader

In [ ]:
# ===== CELL 7 =====
tokenizer = AutoTokenizer.from_pretrained(CONFIG["backbone"])


class PoemDS(Dataset):
    """Tokenizes on the fly (truncated, no padding; padding happens per batch in `collate`).

    NOTE (long poems): truncation=True, max_length=512 simply discards the tail of long poems.
    Recommended upgrade: split each poem into overlapping 512-token windows (stride ~128),
    run the model on each window, and average the logits (or probabilities) per poem.
    """

    def __init__(self, frame: pd.DataFrame, Y: Dict[str, np.ndarray], M: Dict[str, np.ndarray]):
        self.texts = frame[CONFIG["text_col"]].tolist()
        self.Y, self.M = Y, M

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, i: int) -> dict:
        enc = tokenizer(self.texts[i], truncation=True, max_length=CONFIG["max_len"])
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "labels": {t: self.Y[t][i] for t in self.Y},
            "masks": {t: self.M[t][i] for t in self.M},
        }


def collate(items: List[dict]) -> dict:
    """Pad to the longest sequence in the batch and stack labels/masks per task."""
    padded = tokenizer.pad(
        [{"input_ids": it["input_ids"], "attention_mask": it["attention_mask"]} for it in items],
        return_tensors="pt",
    )
    tasks = items[0]["labels"].keys()
    return {
        "input_ids": padded["input_ids"],
        "attention_mask": padded["attention_mask"],
        "labels": {t: torch.tensor(np.stack([it["labels"][t] for it in items]), dtype=torch.float32) for t in tasks},
        "masks": {t: torch.tensor([it["masks"][t] for it in items], dtype=torch.float32) for t in tasks},
    }


g = torch.Generator()
g.manual_seed(CONFIG["seed"])
train_ds = PoemDS(train_df, TRAIN_Y, TRAIN_M)
val_ds = PoemDS(val_df, VAL_Y, VAL_M)
train_dl = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, collate_fn=collate,
                      num_workers=2, pin_memory=USE_AMP, generator=g)
val_dl = DataLoader(val_ds, batch_size=CONFIG["batch_size"] * 2, shuffle=False, collate_fn=collate,
                    num_workers=2, pin_memory=USE_AMP)
print(f"Train batches: {len(train_dl)} Val batches: {len(val_dl)}")


### Cell 8: Model Architecture (VerseLens)

In [ ]:
# ===== CELL 8 =====
class VerseLens(nn.Module):
    """Shared DeBERTa-v3 encoder + one MLP head per task (Linear -> GELU -> Dropout -> Linear)."""

    def __init__(self, backbone: str, task_dims: Dict[str, int], dropout: float = 0.1):
        super().__init__()
        assert all("." not in t for t in task_dims), "Task names must not contain '.' (nn.ModuleDict restriction)."
        # Explicitly enforce float32 master weights so GradScaler can safely unscale gradients
        self.encoder = AutoModel.from_pretrained(backbone, torch_dtype=torch.float32)
        h = self.encoder.config.hidden_size
        self.heads = nn.ModuleDict({
            t: nn.Sequential(nn.Linear(h, h), nn.GELU(), nn.Dropout(dropout), nn.Linear(h, d))
            for t, d in task_dims.items()
        })

    def forward(self, input_ids=None, attention_mask=None, inputs_embeds=None) -> Dict[str, torch.Tensor]:
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, inputs_embeds=inputs_embeds)
        cls = out.last_hidden_state[:, 0]  # [CLS] pooling
        return {t: head(cls) for t, head in self.heads.items()}


set_seed(CONFIG["seed"])
model = VerseLens(CONFIG["backbone"], {t: len(v) for t, v in TASK_VOCAB.items()}).to(DEVICE)
# Enforce float32 master parameters across all layers
model.float()
for p in model.parameters():
    p.data = p.data.float()

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {n_train / 1e6:.1f}M / {n_total / 1e6:.1f}M")
print({t: len(v) for t, v in TASK_VOCAB.items()})


### Cell 9: Optimization, Loss Criteria & Parameter Groups

In [ ]:
# ===== CELL 9 =====
NO_DECAY = ("bias", "LayerNorm.weight", "layer_norm")


def make_groups(named_params, lr: float, wd: float) -> List[dict]:
    """Split parameters into decay / no-decay groups sharing one learning rate."""
    decay = [p for n, p in named_params if not any(nd in n for nd in NO_DECAY)]
    nodecay = [p for n, p in named_params if any(nd in n for nd in NO_DECAY)]
    return [{"params": decay, "lr": lr, "weight_decay": wd},
            {"params": nodecay, "lr": lr, "weight_decay": 0.0}]


param_groups = (make_groups(list(model.encoder.named_parameters()), CONFIG["lr_backbone"], CONFIG["weight_decay"])
                + make_groups(list(model.heads.named_parameters()), CONFIG["lr_heads"], CONFIG["weight_decay"]))
optimizer = torch.optim.AdamW(param_groups)

steps_per_epoch = math.ceil(len(train_dl) / CONFIG["grad_accum"])
total_steps = steps_per_epoch * CONFIG["epochs"]
warmup_steps = int(CONFIG["warmup_ratio"] * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# Modern GradScaler instantiation for PyTorch 2.x
if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
else:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_SCALER)

print(f"Optimizer steps: {total_steps} (warmup {warmup_steps})")

CRITERIA = {t: nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHTS[t].to(DEVICE), reduction="none") for t in TASK_COLS}


def to_device(batch: dict) -> dict:
    """Move every tensor in a collated batch to DEVICE."""
    return {
        "input_ids": batch["input_ids"].to(DEVICE),
        "attention_mask": batch["attention_mask"].to(DEVICE),
        "labels": {t: v.to(DEVICE) for t, v in batch["labels"].items()},
        "masks": {t: v.to(DEVICE) for t, v in batch["masks"].items()},
    }


def task_loss(logits: Dict[str, torch.Tensor], batch: dict) -> Tuple[torch.Tensor, Dict[str, float]]:
    """total = sum_t lambda_t * masked-mean BCE_t (rows with no label for task t are ignored)."""
    total, per_task = 0.0, {}
    for t in TASK_COLS:
        raw = CRITERIA[t](logits[t].float(), batch["labels"][t])           # [B, L]
        m = batch["masks"][t]                                              # [B]
        loss_t = (raw.mean(dim=1) * m).sum() / m.sum().clamp(min=1.0)
        total = total + CONFIG["task_weights"].get(t, 1.0) * loss_t
        per_task[t] = float(loss_t.detach())
    return total, per_task


### Cell 10: Evaluation & Training Loop

In [ ]:
# ===== CELL 10 =====
@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, threshold: Optional[float] = None) -> dict:
    """Per-task micro/macro F1 on labeled rows only, plus mean val loss and avg micro-F1."""
    thr = CONFIG["threshold"] if threshold is None else threshold
    model.eval()
    probs = {t: [] for t in TASK_COLS}
    ys = {t: [] for t in TASK_COLS}
    ms = {t: [] for t in TASK_COLS}
    loss_sum, n = 0.0, 0
    for raw in loader:
        batch = to_device(raw)
        with autocast_ctx():
            logits = model(batch["input_ids"], batch["attention_mask"])
            total, _ = task_loss(logits, batch)
        bs = batch["input_ids"].size(0)
        loss_sum += float(total) * bs
        n += bs
        for t in TASK_COLS:
            probs[t].append(torch.sigmoid(logits[t].float()).cpu().numpy())
            ys[t].append(batch["labels"][t].cpu().numpy())
            ms[t].append(batch["masks"][t].cpu().numpy())

    metrics: Dict[str, dict] = {}
    for t in TASK_COLS:
        p, y, m = np.concatenate(probs[t]), np.concatenate(ys[t]), np.concatenate(ms[t])
        sel = m > 0
        if sel.sum() == 0:
            metrics[t] = {"micro_f1": float("nan"), "macro_f1": float("nan"), "n": 0}
            continue
        yt, yp = y[sel].astype(int), (p[sel] >= thr).astype(int)
        supported = np.where(yt.sum(0) > 0)[0]   # macro over labels that actually occur
        macro = f1_score(yt[:, supported], yp[:, supported], average="macro", zero_division=0) if len(supported) else 0.0
        metrics[t] = {"micro_f1": f1_score(yt, yp, average="micro", zero_division=0),
                      "macro_f1": macro, "n": int(sel.sum())}
    valid = [v["micro_f1"] for v in metrics.values() if v["n"] > 0]
    return {"val_loss": loss_sum / max(n, 1), "tasks": metrics,
            "avg_micro_f1": float(np.mean(valid)) if valid else 0.0}


def train_model(epochs: Optional[int] = None) -> nn.Module:
    """Pure-PyTorch loop: AMP + grad accumulation + clipping; keeps the best epoch by avg micro-F1."""
    epochs = epochs or CONFIG["epochs"]
    accum = CONFIG["grad_accum"]
    best_f1, best_state, best_epoch = -1.0, None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        run_loss, seen = 0.0, 0
        bar = tqdm(train_dl, desc=f"Epoch {epoch}/{epochs}")
        for step, raw in enumerate(bar):
            batch = to_device(raw)
            with autocast_ctx():
                logits = model(batch["input_ids"], batch["attention_mask"])
                total, per_task = task_loss(logits, batch)
            scaler.scale(total / accum).backward()

            if (step + 1) % accum == 0 or (step + 1) == len(train_dl):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            run_loss += float(total.detach())
            seen += 1
            bar.set_postfix(loss=f"{run_loss / seen:.4f}")

        res = evaluate(model, val_dl)
        print(f"\nEpoch {epoch}: train_loss={run_loss / seen:.4f} val_loss={res['val_loss']:.4f}")
        for t, m in res["tasks"].items():
            print(f"  {t:12s} micro-F1={m['micro_f1']:.4f} macro-F1={m['macro_f1']:.4f} (n={m['n']})")
        print(f"  avg micro-F1 = {res['avg_micro_f1']:.4f}")

        if res["avg_micro_f1"] > best_f1:
            best_f1, best_epoch = res["avg_micro_f1"], epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nRestored best epoch {best_epoch} (avg micro-F1 {best_f1:.4f})")
    return model


### Cell 11: Execute Model Training

In [ ]:
# ===== CELL 11 =====
model = train_model()


### Cell 12: Export Checkpoint & Download Zip

In [ ]:
# ===== CELL 12 =====
EXPORT_DIR = "verselens_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

torch.save(model.state_dict(), os.path.join(EXPORT_DIR, "verselens_state_dict.pt"))
tokenizer.save_pretrained(EXPORT_DIR)
with open(os.path.join(EXPORT_DIR, "verselens_meta.json"), "w") as f:
    json.dump({"config": CONFIG, "task_cols": TASK_COLS, "task_vocab": TASK_VOCAB}, f, indent=2)

zip_path = shutil.make_archive("verselens_export", "zip", EXPORT_DIR)
print(f"Saved -> {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab; grab the zip from the file browser instead.")


### Cell 13: Inference & Interpretability (Integrated Gradients Token Attribution)

In [ ]:
# ===== CELL 13 =====
def _encode(poem_text: str) -> dict:
    enc = tokenizer(poem_text, truncation=True, max_length=CONFIG["max_len"], return_tensors="pt")
    return {k: v.to(DEVICE) for k, v in enc.items() if k in ("input_ids", "attention_mask")}


@torch.no_grad()
def predict_probs(poem_text: str) -> Dict[str, np.ndarray]:
    """Sigmoid probabilities for every label of every task."""
    model.eval()
    enc = _encode(poem_text)
    logits = model(enc["input_ids"], enc["attention_mask"])
    return {t: torch.sigmoid(l.float())[0].cpu().numpy() for t, l in logits.items()}


def analyze(poem_text: str, k: int = 3, threshold: Optional[float] = None) -> Dict[str, List[Tuple[str, float]]]:
    """Top-k labels per task whose probability >= threshold."""
    thr = CONFIG["threshold"] if threshold is None else threshold
    result = {}
    for t, p in predict_probs(poem_text).items():
        order = np.argsort(-p)[:k]
        result[t] = [(TASK_VOCAB[t][i], float(p[i])) for i in order if p[i] >= thr]
    return result


def evidence(poem_text: str, task: str, label_idx: int, steps: int = 8,
             verbose: bool = False) -> List[Tuple[str, float]]:
    """Integrated Gradients of the target logit w.r.t. input embeddings (right Riemann sum, `steps` steps).
    Baseline = PAD embedding for ordinary tokens (special tokens kept). Returns (token, attribution)
    sorted high -> low; positive values push the model TOWARD the label."""
    assert task in TASK_COLS, f"Unknown task {task!r}. Available: {list(TASK_COLS)}"
    assert 0 <= label_idx < len(TASK_VOCAB[task]), f"label_idx out of range for {task} (0..{len(TASK_VOCAB[task]) - 1})"
    model.eval()
    enc = _encode(poem_text)
    ids, am = enc["input_ids"], enc["attention_mask"]
    emb_layer = model.encoder.embeddings.word_embeddings

    with torch.no_grad():
        emb = emb_layer(ids)                                                     # [1, T, H]
        pad_emb = emb_layer(torch.tensor([[tokenizer.pad_token_id]], device=DEVICE))
        special = torch.tensor(
            tokenizer.get_special_tokens_mask(ids[0].tolist(), already_has_special_tokens=True),
            dtype=torch.bool, device=DEVICE)
        baseline = torch.where(special[None, :, None], emb, pad_emb.expand_as(emb))

    grad_sum = torch.zeros_like(emb)
    for s in range(1, steps + 1):
        x = (baseline + (s / steps) * (emb - baseline)).detach().requires_grad_(True)
        logit = model(inputs_embeds=x, attention_mask=am)[task][0, label_idx]
        grad_sum += torch.autograd.grad(logit, x)[0]

    attr = ((emb - baseline) * grad_sum / steps).sum(-1)[0].detach().cpu().numpy()  # [T]

    if verbose:  # completeness check: sum(attr) should approximate f(x) - f(baseline)
        with torch.no_grad():
            fx = model(inputs_embeds=emb, attention_mask=am)[task][0, label_idx].item()
            fb = model(inputs_embeds=baseline, attention_mask=am)[task][0, label_idx].item()
        print(f"   IG completeness: sum(attr)={attr.sum():.3f} vs f(x)-f(base)={fx - fb:.3f} (gap shrinks with more steps)")

    toks = tokenizer.convert_ids_to_tokens(ids[0].tolist())
    pairs = [(t.replace("\u2581", "").strip() or t, float(a))
             for t, a, sp in zip(toks, attr, special.cpu().numpy()) if not sp]
    return sorted(pairs, key=lambda z: -z[1])


# ---- Demo: first stanza of Frost's \"Stopping by Woods on a Snowy Evening\" (1923, public domain) ----
DEMO_POEM = """Whose woods these are I think I know.
His house is in the village though;
He will not see me stopping here
To watch his woods fill up with snow."""

print("=== analyze() ===")
print(json.dumps(analyze(DEMO_POEM, k=3), indent=2))

probs = predict_probs(DEMO_POEM)
for task, p in probs.items():
    top = int(np.argmax(p))
    label = TASK_VOCAB[task][top]
    print(f"\n[{task}] top-3 raw scores: " +
          ", ".join(f"{TASK_VOCAB[task][i]}={p[i]:.3f}" for i in np.argsort(-p)[:3]))
    print(f"[{task}] evidence for top label '{label}' (Integrated Gradients, top-12 tokens):")
    for tok, a in evidence(DEMO_POEM, task, top, steps=8, verbose=True)[:12]:
        print(f"   {tok:15s} {a:+.4f}")
